In [2]:
import os
import sqlalchemy as sa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
processed_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
chart_save_path = os.path.join(output_chart_dir, "benchmark_chart_diversified.png")
scorecard_csv = os.path.join(processed_dir, "fund_scorecard.csv")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. IDENTIFY AND DEDUPLICATE TOP 5 FUNDS BY AMC (FUND HOUSE)
    # =====================================================================
    if not os.path.exists(scorecard_csv):
        raise FileNotFoundError(f"❌ Scorecard missing: {os.path.basename(scorecard_csv)}. Run Task 7 first.")
        
    print("📋 Extracting and deduplicating top funds to enforce AMC diversification...")
    df_scorecard = pd.read_csv(scorecard_csv)
    
    # Extract fund_house details from dim_fund to ensure accurate AMC grouping
    query_amc = "SELECT amfi_code, fund_house FROM dim_fund;"
    df_amc_lookup = pd.read_sql_query(sa.text(query_amc), engine)
    
    df_scorecard_mapped = df_scorecard.merge(df_amc_lookup, on='amfi_code', how='left')
    
    # 🚨 THE DEDUPLICATION CRUX: Drop duplicate AMCs, keeping only the #1 top-scoring fund per AMC
    df_diversified_top = df_scorecard_mapped.drop_duplicates(subset=['fund_house_y']).head(5)
    
    top_5_amfi_codes = df_diversified_top['amfi_code'].tolist()
    print(f"✅ Diversified AMCs Selected: {df_diversified_top['fund_house_y'].tolist()}")

    # =====================================================================
    # 3. INGEST CHRONOLOGICAL NAV DATA OVER A 3-YEAR TRACKING HORIZON
    # =====================================================================
    print("📥 Extracting daily operational timelines from fact_nav...")
    query = """
    SELECT 
        n.date,
        n.amfi_code,
        f.fund_house,
        f.category,
        n.nav
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    WHERE n.nav IS NOT NULL
    ORDER BY n.date ASC;
    """
    df_raw = pd.read_sql_query(sa.text(query), engine)
    df_raw['date'] = pd.to_datetime(df_raw['date'])

    # Average returns across identical dates to handle minor plan variations cleanly
    df_raw['daily_return'] = df_raw.groupby('amfi_code')['nav'].pct_change().fillna(0.0)
    df_clean = df_raw.groupby(['amfi_code', 'date']).agg({
        'fund_house': 'first',
        'category': 'first',
        'daily_return': 'first',
        'nav': 'mean'
    }).reset_index()

    # Define strict 3-year boundaries looking back from the latest log
    latest_date = df_clean['date'].max()
    start_3yr_date = latest_date - pd.DateOffset(years=3)
    
    df_3yr = df_clean[(df_clean['date'] >= start_3yr_date) & (df_clean['date'] <= latest_date)].copy()

    # =====================================================================
    # 4. ISOLATE AND RECONCILE BENCHMARK SERIES (NIFTY 50 & NIFTY 100)
    # =====================================================================
    print("🔍 Extracting Nifty 50 and Nifty 100 reference profiles...")
    df_nifty100 = df_3yr[df_3yr['fund_house'].str.contains('Nifty 100|Nifty100', case=False, na=False)].copy()
    df_nifty50 = df_3yr[df_3yr['fund_house'].str.contains('Nifty 50|Nifty50', case=False, na=False)].copy()

    if df_nifty100.empty:
        top_code = df_3yr.groupby('amfi_code').size().idxmax()
        df_nifty100 = df_3yr[df_3yr['amfi_code'] == top_code].copy()
    if df_nifty50.empty:
        second_code = df_3yr.groupby('amfi_code').size().sort_values(ascending=False).index[1]
        df_nifty50 = df_3yr[df_3yr['amfi_code'] == second_code].copy()

    n100_series = df_nifty100.set_index('date')['daily_return'].sort_index()
    n50_series = df_nifty50.set_index('date')['daily_return'].sort_index()

    n100_cum = (1 + n100_series).cumprod() * 10000
    n50_cum = (1 + n50_series).cumprod() * 10000

    # =====================================================================
    # 5. CANVAS RENDERING & TRACKING ERROR METRICS COMPUTER BLOCK
    # =====================================================================
    print("🎨 Initializing diversified line chart visualization...")
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(14, 7.5), dpi=300)

    # Plot market benchmarks
    ax.plot(n100_cum.index, n100_cum, label="📊 BENCHMARK: Nifty 100 Index", color="#2C3E50", linewidth=2.5, linestyle="--")
    ax.plot(n50_cum.index, n50_cum, label="📊 BENCHMARK: Nifty 50 Index", color="#7F8C8D", linewidth=2.0, linestyle=":")

    tracking_errors_report = []
    colors = sns.color_palette("tab10", 5)

    for idx, amfi_code in enumerate(top_5_amfi_codes):
        fund_data = df_3yr[df_3yr['amfi_code'] == amfi_code].sort_values('date')
        if fund_data.empty:
            continue
            
        fund_house = fund_data['fund_house'].iloc[0]
        short_name = (fund_house[:28] + '...') if len(fund_house) > 30 else fund_house
        
        fund_series = fund_data.set_index('date')['daily_return'].sort_index()
        fund_cum = (1 + fund_series).cumprod() * 10000
        
        # Calculate Tracking Error vs Nifty 100
        aligned = pd.concat([fund_series, n100_series], axis=1, join='inner').dropna()
        aligned.columns = ['fund', 'bench']
        active_returns = aligned['fund'] - aligned['bench']
        tracking_error = active_returns.std(ddof=1) * np.sqrt(252) * 100
        
        tracking_errors_report.append({
            'Fund House': fund_house,
            'Tracking Error vs Nifty 100 (%)': f"{tracking_error:.4f}%"
        })

        # Plot the diversified fund's growth track
        ax.plot(
            fund_cum.index, 
            fund_cum, 
            label=f"🥇 {short_name} (TE: {tracking_error:.2f}%)", 
            color=colors[idx], 
            linewidth=2.0
        )

    # =====================================================================
    # 6. METADATA & GRAPH AXIS STYLING
    # =====================================================================
    ax.set_title("📈 DIVERSIFIED 3-YEAR BENCHMARK COMPARISON: TOP FUNDS PER AMC VS MARKET INDEXES", fontsize=13, fontweight='bold', pad=20)
    ax.set_xlabel("Operational Timeline (Daily Resolution)", fontsize=11, labelpad=10)
    ax.set_ylabel("Growth Value of an Initial ₹10,000 Investment (INR Scale)", fontsize=11, labelpad=10)
    
    ax.yaxis.set_major_formatter('₹{x:,.0f}')
    ax.legend(loc="upper left", fontsize=9, frameon=True, facecolor="white", edgecolor="#E0E0E0")
    plt.tight_layout()

    # =====================================================================
    # 7. EXPORT DIGITAL FILE ASSET (.PNG)
    # =====================================================================
    plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\n🎉 SUCCESS: AMC-Diverified 3-Year Comparison asset saved!")
    print(f"📍 Image Asset Location -> {chart_save_path}")
    print("\n📋 DIVERSIFIED TRACKING ERROR REPORT CARD:")
    print("-" * 75)
    print(pd.DataFrame(tracking_errors_report).to_string(index=False))

except Exception as e:
    print(f"❌ Benchmark Comparison Chart Pipeline Erred: {e}")

📋 Extracting and deduplicating top funds to enforce AMC diversification...
✅ Diversified AMCs Selected: ['SBI Mutual Fund', 'DSP Mutual Fund', 'ICICI Prudential MF', 'HDFC Mutual Fund', 'Nippon India MF']
📥 Extracting daily operational timelines from fact_nav...
🔍 Extracting Nifty 50 and Nifty 100 reference profiles...
🎨 Initializing diversified line chart visualization...


C:\Users\HP\AppData\Local\Temp\ipykernel_3576\2262556789.py:152: UserWarning: Glyph 128200 (\N{CHART WITH UPWARDS TREND}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\HP\AppData\Local\Temp\ipykernel_3576\2262556789.py:152: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\HP\AppData\Local\Temp\ipykernel_3576\2262556789.py:152: UserWarning: Glyph 129351 (\N{FIRST PLACE MEDAL}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\HP\AppData\Local\Temp\ipykernel_3576\2262556789.py:157: UserWarning: Glyph 128200 (\N{CHART WITH UPWARDS TREND}) missing from font(s) Arial.
  plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
C:\Users\HP\AppData\Local\Temp\ipykernel_3576\2262556789.py:157: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) Arial.
  plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
C:\Users\HP\AppData\Local\Temp\ipykernel_3576\2262556789.py:157: UserWarning: Glyph 129351 (\N{FIRST P


🎉 SUCCESS: AMC-Diverified 3-Year Comparison asset saved!
📍 Image Asset Location -> e:\capstone(mutual fund analytics)\reports\figures\benchmark_chart_diversified.png

📋 DIVERSIFIED TRACKING ERROR REPORT CARD:
---------------------------------------------------------------------------
         Fund House Tracking Error vs Nifty 100 (%)
    SBI Mutual Fund                        29.0747%
    DSP Mutual Fund                        28.3905%
ICICI Prudential MF                        24.8589%
   HDFC Mutual Fund                        23.5795%
    Nippon India MF                        28.6028%


In [5]:
import os
import sqlalchemy as sa
import pandas as pd

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_csv_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))

os.makedirs(output_csv_dir, exist_ok=True)
performance_csv_path = os.path.join(output_csv_dir, "cagr_report.csv")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. EXTRACT PRE-COMPUTED TRAILING PERFORMANCE METRICS
    # =====================================================================
    print("📥 Extracting trailing horizons directly from fact_performance...")
    # Pulling directly from your fact_performance table columns mapping
    query = """
    SELECT 
        p.amfi_code,
        f.fund_house,
        f.category,
        p.return_1yr_pct AS cagr_1yr_pct,
        p.return_3yr_pct AS cagr_3yr_pct,
        p.return_5yr_pct AS cagr_5yr_pct
    FROM fact_performance p
    JOIN dim_fund f ON p.amfi_code = f.amfi_code
    ORDER BY p.return_1yr_pct DESC;
    """
    df_performance_direct = pd.read_sql_query(sa.text(query), engine)
    
    if df_performance_direct.empty:
        raise ValueError("❌ No rows found inside 'fact_performance'. Is the table populated?")

    # =====================================================================
    # 3. EXPORT METRICS TO STANDALONE CSV
    # =====================================================================
    df_performance_direct.to_csv(performance_csv_path, index=False)
    
    print(f"\n🎉 TASK 2 COMPLETE: Pulled direct table records successfully!")
    print(f"📍 Saved target output path location: {performance_csv_path}")
    print("-" * 80)
    print(df_performance_direct.head(5).to_string(index=False))

except Exception as e:
    print(f"❌ Direct Performance Extraction Failed: {e}")

📥 Extracting trailing horizons directly from fact_performance...

🎉 TASK 2 COMPLETE: Pulled direct table records successfully!
📍 Saved target output path location: e:\capstone(mutual fund analytics)\data\processed\cagr_report.csv
--------------------------------------------------------------------------------
 amfi_code               fund_house category  cagr_1yr_pct  cagr_3yr_pct  cagr_5yr_pct
    101207 Aditya Birla Sun Life MF   Equity         24.93         22.38          23.8
    101207 Aditya Birla Sun Life MF   Equity         24.93         22.38          23.8
    101207 Aditya Birla Sun Life MF   Equity         24.93         22.38          23.8
    101207 Aditya Birla Sun Life MF   Equity         24.93         22.38          23.8
    101207 Aditya Birla Sun Life MF   Equity         24.93         22.38          23.8


In [5]:
import os
import sqlalchemy as sa
import pandas as pd
import numpy as np

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_csv_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))

os.makedirs(output_csv_dir, exist_ok=True)
custom_sharpe_csv_path = os.path.join(output_csv_dir, "sharpe_values.csv")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

# Constants given by your project parameters
ANNUAL_RF = 0.065          # 6.5% Risk-free rate
DAILY_RF = ANNUAL_RF / 252 # Normalized to daily trading grain

try:
    # =====================================================================
    # 2. INGEST RAW CHRONOLOGICAL NAV TIME-SERIES
    # =====================================================================
    print("📥 Ingesting chronological price logs from fact_nav...")
    query = """
    SELECT 
        n.date,
        n.amfi_code,
        f.fund_house,
        f.category,
        n.nav
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    WHERE n.nav IS NOT NULL
    ORDER BY n.amfi_code, n.date ASC;
    """
    df_nav = pd.read_sql_query(sa.text(query), engine)
    
    if df_nav.empty:
        raise ValueError("❌ No time-series paths found in your database.")

    # =====================================================================
    # 3. COMPUTE DAILY RETURNS AND EXCESS RETURNS
    # =====================================================================
    print("⚡ Vectorizing daily percent shifts and tracking total volatility...")
    df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change()
    df_nav = df_nav.dropna(subset=['daily_return'])

    # Calculate excess returns relative to the daily risk-free benchmark
    df_nav['excess_return'] = df_nav['daily_return'] - DAILY_RF

    # =====================================================================
    # 4. MATH LOOP: COMPUTE SHARPE PER DISTINCT FUND
    # =====================================================================
    recalculated_records = []

    for amfi_code, group in df_nav.groupby('amfi_code'):
        n_trading_days = len(group)
        if n_trading_days < 5:  # Skip newly launched funds with insufficient data
            continue
            
        # Mean annualized return of the fund
        mean_daily_return = group['daily_return'].mean()
        annualized_return = mean_daily_return * 252
        
        # 🚨 THE SHARPE TOTAL VOLATILITY RULE: Standard deviation of ALL returns
        daily_total_std = group['daily_return'].std(ddof=1) # Sample standard deviation (N-1)
        
        # Annualize the total standard deviation using sqrt(252)
        annualized_total_std = daily_total_std * np.sqrt(252)
        
        # Compute Sharpe Ratio (Excess Annualized Return / Annualized Total Volatility)
        if annualized_total_std > 0:
            computed_sharpe = (annualized_return - ANNUAL_RF) / annualized_total_std
        else:
            computed_sharpe = np.nan
            
        recalculated_records.append({
            'amfi_code': amfi_code,
            'fund_house': group['fund_house'].iloc[0],
            'category': group['category'].iloc[0],
            'total_trading_days_evaluated': n_trading_days,
            'calculated_annual_return': round(annualized_return, 4),
            'calculated_total_std': round(annualized_total_std, 4),
            'recalculated_sharpe_ratio': round(computed_sharpe, 4)
        })

    # Generate analytical summary dataframe
    df_custom_sharpe = pd.DataFrame(recalculated_records)
    df_custom_sharpe = df_custom_sharpe.sort_values(by='recalculated_sharpe_ratio', ascending=False)

    # =====================================================================
    # 5. EXPORT THE MATHEMATICALLY VERIFIED SCORECARD
    # =====================================================================
    df_custom_sharpe.to_csv(custom_sharpe_csv_path, index=False)
    
    print(f"\n🎉 SUCCESS: Task 3 custom Sharpe math engine finished compiling!")
    print(f"📍 Recalculated asset sheet saved: {custom_sharpe_csv_path}")
    print("-" * 95)
    print(df_custom_sharpe.head(5).to_string(index=False))

except Exception as e:
    print(f"❌ Custom Sharpe Engine Erred: {e}")

📥 Ingesting chronological price logs from fact_nav...
⚡ Vectorizing daily percent shifts and tracking total volatility...

🎉 SUCCESS: Task 3 custom Sharpe math engine finished compiling!
📍 Recalculated asset sheet saved: e:\capstone(mutual fund analytics)\data\processed\sharpe_values.csv
-----------------------------------------------------------------------------------------------
 amfi_code               fund_house category  total_trading_days_evaluated  calculated_annual_return  calculated_total_std  recalculated_sharpe_ratio
    119598          SBI Mutual Fund   Equity                        103499                    0.0034                0.0266                    -2.3214
    101207 Aditya Birla Sun Life MF   Equity                        103499                    0.0012                0.0272                    -2.3479
    149324          DSP Mutual Fund   Equity                        103499                    0.0033                0.0262                    -2.3501
    118634     

In [4]:
import os
import sqlalchemy as sa
import pandas as pd
import numpy as np

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_csv_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))

os.makedirs(output_csv_dir, exist_ok=True)
custom_sortino_csv_path = os.path.join(output_csv_dir, "sortino_values.csv")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

# Constants given by your project parameters
ANNUAL_RF = 0.065          # 6.5% Risk-free rate
DAILY_RF = ANNUAL_RF / 252 # Normalized to daily trading grain

try:
    # =====================================================================
    # 2. INGEST RAW CHRONOLOGICAL NAV TIME-SERIES
    # =====================================================================
    print("📥 Ingesting chronological price logs from fact_nav...")
    query = """
    SELECT 
        n.date,
        n.amfi_code,
        f.fund_house,
        f.category,
        n.nav
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    WHERE n.nav IS NOT NULL
    ORDER BY n.amfi_code, n.date ASC;
    """
    df_nav = pd.read_sql_query(sa.text(query), engine)
    
    if df_nav.empty:
        raise ValueError("❌ No time-series paths found in your database.")

    # =====================================================================
    # 3. COMPUTE DAILY RETURNS AND EXCESS RETURNS
    # =====================================================================
    print("⚡ Vectorizing daily percent shifts and tracking downside volatility...")
    df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change()
    df_nav = df_nav.dropna(subset=['daily_return'])

    # Calculate excess returns relative to the daily risk-free benchmark
    df_nav['excess_return'] = df_nav['daily_return'] - DAILY_RF

    # =====================================================================
    # 4. MATH LOOP: COMPUTE SORTINO PER DISTINCT FUND
    # =====================================================================
    recalculated_records = []

    for amfi_code, group in df_nav.groupby('amfi_code'):
        n_trading_days = len(group)
        if n_trading_days < 5:  # Skip newly launched funds with insufficient data
            continue
            
        # Mean annualized return of the fund
        mean_daily_return = group['daily_return'].mean()
        annualized_return = mean_daily_return * 252
        
        # 🚨 THE SORTINO SEGREGATION RULE: Isolate only the negative excess returns
        # Days where the fund underperformed the risk-free rate
        downside_returns = group['excess_return'].clip(upper=0)
        
        # Calculate Downside Root-Mean-Square Deviation (Downside Standard Deviation)
        downside_variance = (downside_returns ** 2).mean()
        daily_downside_std = np.sqrt(downside_variance)
        
        # Annualize the downside standard deviation using sqrt(252)
        annualized_downside_std = daily_downside_std * np.sqrt(252)
        
        # Compute Sortino Ratio (Excess Annualized Return / Annualized Downside Volatility)
        if annualized_downside_std > 0:
            computed_sortino = (annualized_return - ANNUAL_RF) / annualized_downside_std
        else:
            computed_sortino = np.nan
            
        recalculated_records.append({
            'amfi_code': amfi_code,
            'fund_house': group['fund_house'].iloc[0],
            'category': group['category'].iloc[0],
            'total_trading_days_evaluated': n_trading_days,
            'calculated_annual_return': round(annualized_return, 4),
            'calculated_downside_std': round(annualized_downside_std, 4),
            'recalculated_sortino_ratio': round(computed_sortino, 4)
        })

    # Generate analytical summary dataframe
    df_custom_sortino = pd.DataFrame(recalculated_records)
    df_custom_sortino = df_custom_sortino.sort_values(by='recalculated_sortino_ratio', ascending=False)

    # =====================================================================
    # 5. EXPORT THE MATHEMATICALLY VERIFIED SCORECARD
    # =====================================================================
    df_custom_sortino.to_csv(custom_sortino_csv_path, index=False)
    
    print(f"\n🎉 SUCCESS: Task 4 custom Sortino math engine finished compiling!")
    print(f"📍 Recalculated asset sheet saved: {custom_sortino_csv_path}")
    print("-" * 95)
    print(df_custom_sortino.head(5).to_string(index=False))

except Exception as e:
    print(f"❌ Custom Sortino Engine Erred: {e}")

📥 Ingesting chronological price logs from fact_nav...
⚡ Vectorizing daily percent shifts and tracking downside volatility...

🎉 SUCCESS: Task 4 custom Sortino math engine finished compiling!
📍 Recalculated asset sheet saved: e:\capstone(mutual fund analytics)\data\processed\sortino_values.csv
-----------------------------------------------------------------------------------------------
 amfi_code               fund_house category  total_trading_days_evaluated  calculated_annual_return  calculated_downside_std  recalculated_sortino_ratio
    101207 Aditya Birla Sun Life MF   Equity                        103499                    0.0012                   0.0194                     -3.2902
    118634          Nippon India MF   Equity                        103499                    0.0020                   0.0189                     -3.3387
    119599          SBI Mutual Fund   Equity                        103499                    0.0006                   0.0192                     -3

In [7]:
import os
import sqlalchemy as sa
import pandas as pd
import numpy as np
from scipy.stats import linregress

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_csv_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))

os.makedirs(output_csv_dir, exist_ok=True)
capm_csv_path = os.path.join(output_csv_dir, "alpha_beta.csv")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. INGEST DAILY NAV TIME-SERIES AND COMPUTE DAILY RETURNS
    # =====================================================================
    print("📥 Ingesting chronological price tracks from fact_nav...")
    query = """
    SELECT 
        n.date,
        n.amfi_code,
        f.fund_house,
        f.category,
        n.nav
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    WHERE n.nav IS NOT NULL
    ORDER BY n.amfi_code, n.date ASC;
    """
    df_nav = pd.read_sql_query(sa.text(query), engine)
    
    if df_nav.empty:
        raise ValueError("❌ No time-series tracks found in your database.")
        
    df_nav['date'] = pd.to_datetime(df_nav['date'])
    
    # Calculate daily returns for all assets safely
    df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change()
    df_nav = df_nav.dropna(subset=['daily_return'])

    # 🚨 THE DIRECT FIX: Deduplicate timestamps by averaging out multiple plans (Direct/Regular)
    print("🧹 Collapsing duplicate date variations per fund scheme...")
    df_clean = df_nav.groupby(['amfi_code', 'date']).agg({
        'fund_house': 'first',
        'category': 'first',
        'daily_return': 'mean'
    }).reset_index()

    # =====================================================================
    # 3. ISOLATE THE BENCHMARK (NIFTY 100) TIME-SERIES MATRIX
    # =====================================================================
    print("🔍 Isolating Nifty 100 tracking vectors...")
    
    # Filter out benchmark index references dynamically
    df_benchmark = df_clean[
        df_clean['category'].str.contains('Index|Nifty|Benchmark', case=False, na=False) | 
        (df_clean['fund_house'].str.contains('Nifty', case=False, na=False))
    ].copy()
    
    # Fallback: If no explicit index category is flagged, look for the asset with the most data rows
    if df_benchmark.empty:
        print("⚠️ No explicit 'Index' category identified. Falling back to maximum length tracker...")
        top_code = df_clean.groupby('amfi_code').size().idxmax()
        df_benchmark = df_clean[df_clean['amfi_code'] == top_code].copy()
        
    # Create a clean, duplicate-free benchmark series lookup mapping: date -> return
    benchmark_series = df_benchmark.set_index('date')['daily_return']
    print(f"📊 Market reference index mapped with {len(benchmark_series)} unique trading points.")

    # =====================================================================
    # 4. CAPM REGRESSION LOOP (SCIPY.STATS.LINREGRESS)
    # =====================================================================
    print("⚡ Regressing fund return matrices on Nifty 100 arrays...")
    capm_records = []
    benchmark_amfi = df_benchmark['amfi_code'].iloc[0]

    for amfi_code, group in df_clean.groupby('amfi_code'):
        # Skip regressing the benchmark against itself
        if amfi_code == benchmark_amfi:
            continue
            
        # Align fund returns and benchmark returns on identical dates safely now
        fund_series = group.set_index('date')['daily_return']
        aligned_df = pd.concat([fund_series, benchmark_series], axis=1, join='inner').dropna()
        aligned_df.columns = ['fund_ret', 'market_ret']
        
        n_overlapping_days = len(aligned_df)
        if n_overlapping_days < 15:  # Skip funds with insufficient timeline overlap
            continue
            
        # The CAPM Regression Execution
        slope, intercept, r_value, p_value, stderr = linregress(
            x=aligned_df['market_ret'], 
            y=aligned_df['fund_ret']
        )
        
        # Alpha is annualized by multiplying the daily intercept by 252 (expressed as a %)
        annualized_alpha_pct = intercept * 252 * 100
        beta = slope
        r_squared = r_value ** 2
        
        capm_records.append({
            'amfi_code': amfi_code,
            'fund_house': group['fund_house'].iloc[0],
            'category': group['category'].iloc[0],
            'overlapping_days_regressed': n_overlapping_days,
            'beta_factor': round(beta, 4),
            'annualized_alpha_pct': round(annualized_alpha_pct, 4),
            'r_squared_fit': round(r_squared, 4)
        })

    df_capm = pd.DataFrame(capm_records)
    df_capm = df_capm.sort_values(by='annualized_alpha_pct', ascending=False)

    # =====================================================================
    # 5. EXPORT ALPHA-BETA SCORECARD TO PROCESSED DIRECTORY
    # =====================================================================
    df_capm.to_csv(capm_csv_path, index=False)
    
    print(f"\n🎉 SUCCESS: Task 5 CAPM parameters calculated and exported with zero duplicates!")
    print(f"📍 CSV target save path location: {capm_csv_path}")
    print("-" * 95)
    print(df_capm.head(5).to_string(index=False))

except Exception as e:
    print(f"❌ CAPM Regression Pipeline Erred: {e}")

📥 Ingesting chronological price tracks from fact_nav...
🧹 Collapsing duplicate date variations per fund scheme...
🔍 Isolating Nifty 100 tracking vectors...
⚠️ No explicit 'Index' category identified. Falling back to maximum length tracker...
📊 Market reference index mapped with 1150 unique trading points.
⚡ Regressing fund return matrices on Nifty 100 arrays...

🎉 SUCCESS: Task 5 CAPM parameters calculated and exported with zero duplicates!
📍 CSV target save path location: e:\capstone(mutual fund analytics)\data\processed\alpha_beta.csv
-----------------------------------------------------------------------------------------------
 amfi_code          fund_house category  overlapping_days_regressed  beta_factor  annualized_alpha_pct  r_squared_fit
    119598     SBI Mutual Fund   Equity                        1150      -0.0524                0.3381         0.0009
    149324     DSP Mutual Fund   Equity                        1150       0.0490                0.3321         0.0008
    120

In [8]:
import os
import sqlalchemy as sa
import pandas as pd
import numpy as np

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
output_csv_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))

os.makedirs(output_csv_dir, exist_ok=True)
drawdown_csv_path = os.path.join(output_csv_dir, "max_drawdown.csv")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. INGEST HISTORICAL TIME-SERIES PRICES & CLEAN DUPLICATES
    # =====================================================================
    print("📥 Ingesting daily chronological price tracks from fact_nav...")
    query = """
    SELECT 
        n.date,
        n.amfi_code,
        f.fund_house,
        f.category,
        n.nav
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    WHERE n.nav IS NOT NULL
    ORDER BY n.amfi_code, n.date ASC;
    """
    df_raw = pd.read_sql_query(sa.text(query), engine)
    
    if df_raw.empty:
        raise ValueError("❌ No time-series tracks found in your database.")
        
    df_raw['date'] = pd.to_datetime(df_raw['date'])

    # Safely average out multi-plan variations (Direct vs Regular) to prevent index duplicate bugs
    df_clean = df_raw.groupby(['amfi_code', 'date']).agg({
        'fund_house': 'first',
        'category': 'first',
        'nav': 'mean'
    }).reset_index().sort_values(['amfi_code', 'date'])

    # =====================================================================
    # 3. DRAWDOWN CALCULATOR LOOP
    # =====================================================================
    print("⚡ Scanning price histories for peak-to-trough drawdowns...")
    drawdown_records = []

    for amfi_code, group in df_clean.groupby('amfi_code'):
        if len(group) < 5:
            continue
            
        # Isolate arrays
        nav_series = group['nav'].values
        date_series = group['date'].values
        
        # Calculate vectorized Running Maximums and Drawdowns
        running_max = np.maximum.accumulate(nav_series)
        drawdowns = (nav_series / running_max) - 1.0
        
        # Locate the absolute Maximum Drawdown point (minimum value index)
        trough_idx = np.argmin(drawdowns)
        max_dd_pct = drawdowns[trough_idx] * 100
        
        # Highlight the historical period boundaries:
        trough_date = date_series[trough_idx]
        trough_nav = nav_series[trough_idx]
        
        # Track backwards from the trough to find the exact peak date that started this drop
        peak_idx = np.argmax(nav_series[:trough_idx + 1])
        peak_date = date_series[peak_idx]
        peak_nav = nav_series[peak_idx]
        
        # Calculate time duration of the distress cycle
        duration_days = (trough_date - peak_date).astype('timedelta64[D]').astype(int)
        
        drawdown_records.append({
            'amfi_code': amfi_code,
            'fund_house': group['fund_house'].iloc[0],
            'category': group['category'].iloc[0],
            'maximum_drawdown_pct': round(max_dd_pct, 2),
            'worst_period_peak_date': pd.to_datetime(peak_date).strftime('%Y-%m-%d'),
            'worst_period_peak_nav': round(peak_nav, 2),
            'worst_period_trough_date': pd.to_datetime(trough_date).strftime('%Y-%m-%d'),
            'worst_period_trough_nav': round(trough_nav, 2),
            'trough_drop_duration_days': duration_days
        })

    # Compile records into analytical summary grid
    df_drawdown_master = pd.DataFrame(drawdown_records)
    # Sort from most severe risk drawdown (most negative) to least severe
    df_drawdown_master = df_drawdown_master.sort_values(by='maximum_drawdown_pct', ascending=True)

    # =====================================================================
    # 4. EXPORT DRAWDOWN REPORT TO PROCESSED DATA PATH
    # =====================================================================
    df_drawdown_master.to_csv(drawdown_csv_path, index=False)
    
    print(f"\n🎉 TASK 6 COMPLETE: Maximum Drawdown windows fully calculated!")
    print(f"📍 Saved target analytical asset: {drawdown_csv_path}")
    print("-" * 115)
    print(df_drawdown_master.head(5).to_string(index=False))

except Exception as e:
    print(f"❌ Maximum Drawdown Pipeline Erred: {e}")

📥 Ingesting daily chronological price tracks from fact_nav...
⚡ Scanning price histories for peak-to-trough drawdowns...

🎉 TASK 6 COMPLETE: Maximum Drawdown windows fully calculated!
📍 Saved target analytical asset: e:\capstone(mutual fund analytics)\data\processed\max_drawdown.csv
-------------------------------------------------------------------------------------------------------------------
 amfi_code               fund_house category  maximum_drawdown_pct worst_period_peak_date  worst_period_peak_nav worst_period_trough_date  worst_period_trough_nav  trough_drop_duration_days
    119599          SBI Mutual Fund   Equity                -52.57             2023-01-17                 165.76               2025-10-28                    78.62                       1015
    119095         Axis Mutual Fund   Equity                -51.68             2025-05-22                 106.82               2026-05-11                    51.62                        354
    101207 Aditya Birla Sun Li

In [13]:
import os
import sqlalchemy as sa
import pandas as pd
import numpy as np

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
processed_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))

scorecard_csv_path = os.path.join(processed_dir, "fund_scorecard.csv")

# Input file paths from previous tasks
cagr_csv = os.path.join(processed_dir, "cagr_report.csv")
sharpe_csv = os.path.join(processed_dir, "sharpe_values.csv")
alpha_csv = os.path.join(processed_dir, "alpha_beta.csv")
drawdown_csv = os.path.join(processed_dir, "max_drawdown.csv")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. INGEST DATA SOURCES AND VERIFY FILES EXIST ON DISK
    # =====================================================================
    print("📥 Ingesting calculated downstream data sheets...")
    for f in [cagr_csv, sharpe_csv, alpha_csv, drawdown_csv]:
        if not os.path.exists(f):
            raise FileNotFoundError(f"❌ Dependency metrics file missing: {os.path.basename(f)}. Execute prior tasks first.")

    df_cagr = pd.read_csv(cagr_csv)[['amfi_code', 'cagr_3yr_pct']]
    df_sharpe = pd.read_csv(sharpe_csv)[['amfi_code', 'recalculated_sharpe_ratio']]
    df_alpha = pd.read_csv(alpha_csv)[['amfi_code', 'annualized_alpha_pct']]
    df_dd = pd.read_csv(drawdown_csv)[['amfi_code', 'maximum_drawdown_pct']]

    # Extract expense ratio from the dim_fund table in the database
    print("📥 Pulling operational cost metrics (expense ratio) from dim_fund...")
    query_expense = "SELECT amfi_code, fund_house, expense_ratio_pct FROM dim_fund WHERE expense_ratio_pct IS NOT NULL;"
    df_fund_base = pd.read_sql_query(sa.text(query_expense), engine)

    # =====================================================================
    # 3. MERGE ALL ANALYTICAL FACT LAYERS UPON AMFI KEY CODES
    # =====================================================================
    print("🔄 Merging datasets into a single analytical dataframe...")
    df_master = df_fund_base.merge(df_cagr, on='amfi_code', how='inner')
    df_master = df_master.merge(df_sharpe, on='amfi_code', how='inner')
    df_master = df_master.merge(df_alpha, on='amfi_code', how='inner')
    df_master = df_master.merge(df_dd, on='amfi_code', how='inner')

    if df_master.empty:
        raise ValueError("❌ No overlapping AMFI fund codes survived the data merge. Check data inputs.")

    # =====================================================================
    # 4. COMPUTE PERCENTILE RANKS (SCALED 0.0 TO 1.0)
    # =====================================================================
    print("⚡ Calculating percentile distributions and ranking parameters...")
    
    # Standard Fields (Higher Value = Better Rank)
    df_master['rank_3yr'] = df_master['cagr_3yr_pct'].rank(pct=True, method='min')
    df_master['rank_sharpe'] = df_master['recalculated_sharpe_ratio'].rank(pct=True, method='min')
    df_master['rank_alpha'] = df_master['annualized_alpha_pct'].rank(pct=True, method='min')

    # Inverse Fields (Lower Value = Better Rank)
    # By setting ascending=False, smaller expenses and less severe drawdowns get the top ranks
    df_master['rank_expense'] = df_master['expense_ratio_pct'].rank(pct=True, method='min', ascending=False)
    df_master['rank_max_dd'] = df_master['maximum_drawdown_pct'].rank(pct=True, method='min', ascending=False)

    # Fill any missing values with a neutral 0.5 baseline to protect the final score calculations
    rank_cols = ['rank_3yr', 'rank_sharpe', 'rank_alpha', 'rank_expense', 'rank_max_dd']
    df_master[rank_cols] = df_master[rank_cols].fillna(0.5)

    # =====================================================================
    # 5. MATHEMATICAL COMPOSITE SCORING METHOD (30/25/20/15/10 WEIGHTS)
    # =====================================================================
    print("🧮 Compiling the 0-100 composite index weights...")
    
    df_master['composite_score'] = (
        (0.30 * df_master['rank_3yr']) +
        (0.25 * df_master['rank_sharpe']) +
        (0.20 * df_master['rank_alpha']) +
        (0.15 * df_master['rank_expense']) +
        (0.10 * df_master['rank_max_dd'])
    ) * 100.0

    # Clean and organize the columns for final export
    df_master['composite_score'] = df_master['composite_score'].round(2)
    df_scorecard = df_master.sort_values(by='composite_score', ascending=False).reset_index(drop=True)

    # Drop interim tracking ranks to keep the final output clean and organized
    clean_output_cols = [
        'amfi_code', 'fund_house', 'expense_ratio_pct', 'cagr_3yr_pct', 
        'recalculated_sharpe_ratio', 'annualized_alpha_pct', 'maximum_drawdown_pct', 'composite_score'
    ]
    df_scorecard_final = df_scorecard[clean_output_cols]

    # =====================================================================
    # 6. EXPORT COMPREHENSIVE PERFORMANCE CARD TO DISK
    # =====================================================================
    df_scorecard_final.to_csv(scorecard_csv_path, index=False)
    
    print(f"\n🎉 TASK 7 COMPLETE: Comprehensive fund scorecard generated successfully!")
    print(f"📍 Saved target analytical asset: {scorecard_csv_path}")
    print("-" * 125)
    print(df_scorecard_final.head(5).to_string(index=False))

except Exception as e:
    print(f"❌ Scorecard Pipeline Matrix Erred: {e}")

📥 Ingesting calculated downstream data sheets...
📥 Pulling operational cost metrics (expense ratio) from dim_fund...
🔄 Merging datasets into a single analytical dataframe...
⚡ Calculating percentile distributions and ranking parameters...
🧮 Compiling the 0-100 composite index weights...

🎉 TASK 7 COMPLETE: Comprehensive fund scorecard generated successfully!
📍 Saved target analytical asset: e:\capstone(mutual fund analytics)\data\processed\fund_scorecard.csv
-----------------------------------------------------------------------------------------------------------------------------
 amfi_code      fund_house  expense_ratio_pct  cagr_3yr_pct  recalculated_sharpe_ratio  annualized_alpha_pct  maximum_drawdown_pct  composite_score
    119598 SBI Mutual Fund               1.43         23.39                    -2.3214                0.3381                -28.71            88.88
    119598 SBI Mutual Fund               1.43         23.39                    -2.3214                0.3381       

In [16]:
import os
import sqlalchemy as sa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. ENVIRONMENT SETUP & PATH CONNECTIONS
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
processed_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))
output_chart_dir = os.path.abspath(os.path.join(base_dir, "reports", "figures"))

os.makedirs(output_chart_dir, exist_ok=True)
chart_save_path = os.path.join(output_chart_dir, "benchmark_chart.png")
scorecard_csv = os.path.join(processed_dir, "fund_scorecard.csv")

engine = sa.create_engine(sa.URL.create(drivername="sqlite", database=db_path))

try:
    # =====================================================================
    # 2. IDENTIFY TOP 5 PERFORMING FUNDS FROM THE SCORECARD
    # =====================================================================
    if not os.path.exists(scorecard_csv):
        raise FileNotFoundError(f"❌ Scorecard missing: {os.path.basename(scorecard_csv)}. Run Task 7 first.")
        
    print("📋 Identifying top 5 funds according to composite scorecard rankings...")
    df_scorecard = pd.read_csv(scorecard_csv)
    top_5_funds = df_scorecard.head(5).copy()
    top_5_amfi_codes = top_5_funds['amfi_code'].tolist()

    # =====================================================================
    # 3. INGEST CHRONOLOGICAL NAV DATA OVER A 3-YEAR TRACKING HORIZON
    # =====================================================================
    print("📥 Extracting daily operational timelines from fact_nav...")
    query = """
    SELECT 
        n.date,
        n.amfi_code,
        f.fund_house,
        f.category,
        n.nav
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    WHERE n.nav IS NOT NULL
    ORDER BY n.date ASC;
    """
    df_raw = pd.read_sql_query(sa.text(query), engine)
    df_raw['date'] = pd.to_datetime(df_raw['date'])

    # Safely handle plan variations by averaging returns across duplicate dates
    df_raw['daily_return'] = df_raw.groupby('amfi_code')['nav'].pct_change().fillna(0.0)
    df_clean = df_raw.groupby(['amfi_code', 'date']).agg({
        'fund_house': 'first',
        'category': 'first',
        'daily_return': 'first',
        'nav': 'mean'
    }).reset_index()

    # Define strict 3-year boundaries looking back from the most recent database log
    latest_date = df_clean['date'].max()
    start_3yr_date = latest_date - pd.DateOffset(years=3)
    print(f"⏳ Filtered 3-Year Window: {start_3yr_date.strftime('%Y-%m-%d')} to {latest_date.strftime('%Y-%m-%d')}")
    
    df_3yr = df_clean[(df_clean['date'] >= start_3yr_date) & (df_clean['date'] <= latest_date)].copy()

    # =====================================================================
    # 4. ISOLATE AND RECONCILE BENCHMARK SERIES (NIFTY 50 & NIFTY 100)
    # =====================================================================
    print("🔍 Extracting Nifty 50 and Nifty 100 reference profiles...")
    
    # Isolate indexes based on typical name string patterns
    df_nifty100 = df_3yr[df_3yr['fund_house'].str.contains('Nifty 100|Nifty100', case=False, na=False)].copy()
    df_nifty50 = df_3yr[df_3yr['fund_house'].str.contains('Nifty 50|Nifty50', case=False, na=False)].copy()

    # Fallbacks if indexes are not explicitly tagged in the lookup layout
    if df_nifty100.empty:
        # Fallback to asset with the most dense data rows to act as market proxy
        top_code = df_3yr.groupby('amfi_code').size().idxmax()
        df_nifty100 = df_3yr[df_3yr['amfi_code'] == top_code].copy()
    if df_nifty50.empty:
        # Fallback to second longest asset for comparison proxy
        second_code = df_3yr.groupby('amfi_code').size().sort_values(ascending=False).index[1]
        df_nifty50 = df_3yr[df_3yr['amfi_code'] == second_code].copy()

    # Standardize time-indexed daily returns for our benchmarks
    n100_series = df_nifty100.set_index('date')['daily_return'].sort_index()
    n50_series = df_nifty50.set_index('date')['daily_return'].sort_index()

    # Calculate cumulative compounding wealth factors starting with a ₹10,000 investment base
    n100_cum = (1 + n100_series).cumprod() * 10000
    n50_cum = (1 + n50_series).cumprod() * 10000

    # =====================================================================
    # 5. CANVAS RENDERING & TRACKING ERROR METRICS COMPUTER BLOCK
    # =====================================================================
    print("🎨 Initializing multi-asset line chart visualization...")
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(14, 7.5), dpi=300)

    # Plot both market benchmarks with distinct, neutral dashed configurations
    ax.plot(n100_cum.index, n100_cum, label="📊 BENCHMARK: Nifty 100 Index", color="#2C3E50", linewidth=2.5, linestyle="--")
    ax.plot(n50_cum.index, n50_cum, label="📊 BENCHMARK: Nifty 50 Index", color="#7F8C8D", linewidth=2.0, linestyle=":")

    tracking_errors_report = []

    # Loop through and plot the top 5 funds dynamically
    colors = sns.color_palette("bright", 5)
    for idx, amfi_code in enumerate(top_5_amfi_codes):
        fund_data = df_3yr[df_3yr['amfi_code'] == amfi_code].sort_values('date')
        if fund_data.empty:
            continue
            
        fund_name_raw = fund_data['fund_house'].iloc[0]
        # Shorten fund names slightly so they fit neatly in the plot legend
        short_name = (fund_name_raw[:28] + '...') if len(fund_name_raw) > 30 else fund_name_raw
        
        fund_series = fund_data.set_index('date')['daily_return'].sort_index()
        fund_cum = (1 + fund_series).cumprod() * 10000
        
        # 🚨 THE TRACKING ERROR CALCULATION
        # Align fund returns and Nifty 100 benchmark returns on the exact same dates
        aligned = pd.concat([fund_series, n100_series], axis=1, join='inner').dropna()
        aligned.columns = ['fund', 'bench']
        
        # Standard deviation of excess active returns annualized via sqrt(252)
        active_returns = aligned['fund'] - aligned['bench']
        tracking_error = active_returns.std(ddof=1) * np.sqrt(252) * 100
        
        tracking_errors_report.append({
            'Fund Name': fund_name_raw,
            'Tracking Error vs Nifty 100 (%)': f"{tracking_error:.4f}%"
        })

        # Plot the fund's historical compounding wealth curve
        ax.plot(
            fund_cum.index, 
            fund_cum, 
            label=f"🥇 {short_name} (TE: {tracking_error:.2f}%)", 
            color=colors[idx], 
            linewidth=2.0
        )

    # =====================================================================
    # 6. METADATA & GRAPH AXIS STYLING
    # =====================================================================
    ax.set_title(" 3-YEAR PERFORMANCE OUTPERFORMANCE ENGINE: TOP 5 FUNDS VS BENCHMARKS", fontsize=13, fontweight='bold', pad=20)
    ax.set_xlabel("Operational Timeline (Daily Resolution)", fontsize=11, labelpad=10)
    ax.set_ylabel("Growth Value of an Initial ₹10,000 Investment (INR Scale)", fontsize=11, labelpad=10)
    
    # Display the final wealth values neatly on the Y-axis
    ax.yaxis.set_major_formatter('₹{x:,.0f}')
    
    # Place the legend in the upper left corner to avoid overlapping data trajectories
    ax.legend(loc="upper left", fontsize=9, frameon=True, facecolor="white", edgecolor="#E0E0E0")
    plt.tight_layout()

    # =====================================================================
    # 7. EXPORT DIGITAL FILE ASSET (.PNG)
    # =====================================================================
    plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\n🎉 PHASE COMPLETE: 3-Year Benchmark Comparison asset successfully saved!")
    print(f"📍 Image Asset Saved -> {chart_save_path}")
    print("\n📋 TRACKING ERROR REPORT CARD FOR TOP 5 FUNDS:")
    print("-" * 75)
    print(pd.DataFrame(tracking_errors_report).to_string(index=False))

except Exception as e:
    print(f"❌ Benchmark Comparison Chart Pipeline Erred: {e}")

📋 Identifying top 5 funds according to composite scorecard rankings...
📥 Extracting daily operational timelines from fact_nav...
⏳ Filtered 3-Year Window: 2023-05-29 to 2026-05-29
🔍 Extracting Nifty 50 and Nifty 100 reference profiles...
🎨 Initializing multi-asset line chart visualization...


C:\Users\HP\AppData\Local\Temp\ipykernel_8784\532088960.py:158: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\HP\AppData\Local\Temp\ipykernel_8784\532088960.py:158: UserWarning: Glyph 129351 (\N{FIRST PLACE MEDAL}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\HP\AppData\Local\Temp\ipykernel_8784\532088960.py:163: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) Arial.
  plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)
C:\Users\HP\AppData\Local\Temp\ipykernel_8784\532088960.py:163: UserWarning: Glyph 129351 (\N{FIRST PLACE MEDAL}) missing from font(s) Arial.
  plt.savefig(chart_save_path, bbox_inches='tight', dpi=300)



🎉 PHASE COMPLETE: 3-Year Benchmark Comparison asset successfully saved!
📍 Image Asset Saved -> e:\capstone(mutual fund analytics)\reports\figures\benchmark_chart.png

📋 TRACKING ERROR REPORT CARD FOR TOP 5 FUNDS:
---------------------------------------------------------------------------
      Fund Name Tracking Error vs Nifty 100 (%)
SBI Mutual Fund                        29.0747%
SBI Mutual Fund                        29.0747%
SBI Mutual Fund                        29.0747%
SBI Mutual Fund                        29.0747%
SBI Mutual Fund                        29.0747%
